In [ ]:
# Import some libraries and especially load environment variables
from langchain_openai  import AzureChatOpenAI
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv()) # read local .env file
# set an environment varibale
os.environ["LANGCHAIN_PROJECT"] = "01_basic"

print(os.getenv("AZURE_ENDPOINT"))


In [ ]:
llm = AzureChatOpenAI(
	temperature=0,
    openai_api_version="2024-10-21",
    deployment_name="GPT-4o-mini", #Deployment name
    azure_endpoint=os.environ["OPENAI_API_BASE"],
    model_name="GPT-4o-mini"
)

In [ ]:
# I the import my custom function to work with audio and videp
from plugins.AudioVideo import AudioVideo
from plugins.Summarization import Summarization

summarization = Summarization()
audio_video = AudioVideo()

# # you can test the function
# audio_video.extract_audio("C:\\temp\\300.mp4")
# audio_video.extract_audio("/Users/gianmariaricci/develop/montaggi/RDP.mp4");

In [ ]:
# Then create tools list
from langchain.tools import Tool

tools = [
    Tool.from_function(
        func=audio_video.extract_audio,
        name="ExtractAudio",
        description="extract audio in wav format from an mp4 file",
    ),
    Tool.from_function(
        func=audio_video.transcript_timeline,
        name="TranscriptTimeline",
        description="Transcript audio from a wav file to a full transcription",
    ),
    Tool.from_function(
        func=summarization.summarize_timeline,
        name="SummarizeTimeline",
        description="Take a full transcription and create a summarized timeline.",
    ),
]

In [ ]:
from langchain import hub

# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages

In [ ]:
# Finally you can create the agent
from langchain.agents import create_tool_calling_agent
agent = create_tool_calling_agent(tools=tools, llm=llm, prompt=prompt)

In [ ]:
# Everthing is ready to start the conversation
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
result = agent_executor.invoke({"input": "I need a summarized timeline from video c:\\temp\\ssh.mp4"})

In [ ]:
from pprint import pprint 
pprint(result)